In [ ]:
import pyarrow.parquet as pq
import pyarrow as pa

evals_pq  = r"./lichess-eval-db/evals.parquet"
sample_pq = r"5m-evals.parquet"

TOTAL_ROWS = 5_000_000
CHUNK_SIZE = 1_000_000

pf = pq.ParquetFile(evals_pq)
chunks = []
rows_read = 0

for batch in pf.iter_batches(batch_size=CHUNK_SIZE):
    chunks.append(pa.Table.from_batches([batch]))
    rows_read += len(batch)
    print(f"Read {rows_read:,} / {TOTAL_ROWS:,} rows")
    if rows_read >= TOTAL_ROWS:
        break

table = pa.concat_tables(chunks).slice(0, TOTAL_ROWS)
pq.write_table(table, sample_pq)
print(f"Done — wrote {len(table):,} rows to {sample_pq}")
